In [18]:
import pickle # Load refs and annotations
import json
import os
import pandas as pd
import numpy as np
import pprint
import json
import cv2
import random

from typing import Any, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.utils.tensorboard import SummaryWriter

import torchvision
import torchvision.transforms as transforms
from torchvision.utils import draw_bounding_boxes
from torchvision import models
import torchmetrics

import pytorch_lightning as pl
from pytorch_lightning.utilities.types import STEP_OUTPUT

from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import CLIPProcessor, CLIPModel

from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import clip
from ultralytics import YOLO
from PIL import Image, ImageDraw

from ipywidgets import FloatProgress
import math 
from torch.nn.modules.batchnorm import _BatchNorm
from torchvision.ops import box_convert

In [188]:
import torch
image = torch.zeros((1, 3, 224, 224)).float()
bbox = torch.FloatTensor([[20, 30, 100, 200], [50, 100, 150, 200]]) # [y1, x1, y2, x2] format
labels = torch.LongTensor([6, 8]) # 0 represents background
sub_sample = 16

In [189]:
import torchvision
dummy_img = torch.zeros((1, 3, 224, 224)).float()
print(dummy_img)
#Out: torch.Size([1, 3, 800, 800])

tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]],

         [[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]]])


In [190]:
model = torchvision.models.resnet50(pretrained=True)

/opt/anaconda3/envs/bagigio/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/anaconda3/envs/bagigio/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [191]:
fe = list(model.children())


In [192]:
req_features = []
fee = []
k = dummy_img.clone()
for i in fe:
    k = i(k)
    if k.size()[2] < 224//16:
        break
    req_features.append(i)
    out_channels = k.size()[1]
print(len(req_features)) #30
print(out_channels) # 512

7
1024


In [193]:
faster_rcnn_fe_extractor = nn.Sequential(*req_features)

In [194]:
out_map = faster_rcnn_fe_extractor(image)
print(out_map.size())

torch.Size([1, 1024, 14, 14])


In [149]:
224/14

16.0

In [196]:
ratios = [0.5, 1, 2]
anchor_scales = [8, 16, 32]
anchor_base = np.zeros((len(ratios) * len(anchor_scales), 4), dtype=np.float32)
print(anchor_base)
print(anchor_base.shape)

[[          0           0           0           0]
 [          0           0           0           0]
 [          0           0           0           0]
 [          0           0           0           0]
 [          0           0           0           0]
 [          0           0           0           0]
 [          0           0           0           0]
 [          0           0           0           0]
 [          0           0           0           0]]
(9, 4)


In [197]:
ctr_y = sub_sample / 2.
ctr_x = sub_sample / 2.
print(ctr_y, ctr_x)
# Out: (8, 8)
for i in range(len(ratios)):
  for j in range(len(anchor_scales)):
    h = sub_sample * anchor_scales[j] * np.sqrt(ratios[i])
    w = sub_sample * anchor_scales[j] * np.sqrt(1./ ratios[i])
    index = i * len(anchor_scales) + j
    anchor_base[index, 0] = ctr_y - h / 2.
    anchor_base[index, 1] = ctr_x - w / 2.
    anchor_base[index, 2] = ctr_y + h / 2.
    anchor_base[index, 3] = ctr_x + w / 2.
print(anchor_base)
print(anchor_base.shape)

8.0 8.0
[[    -37.255      -82.51      53.255       98.51]
 [     -82.51     -173.02       98.51      189.02]
 [    -173.02     -354.04      189.02      370.04]
 [        -56         -56          72          72]
 [       -120        -120         136         136]
 [       -248        -248         264         264]
 [     -82.51     -37.255       98.51      53.255]
 [    -173.02      -82.51      189.02       98.51]
 [    -354.04     -173.02      370.04      189.02]]
(9, 4)


In [198]:
fe_size = (224//16)
ctr_x = np.arange(16, (fe_size+1) * 16, 16)
ctr_y = np.arange(16, (fe_size+1) * 16, 16)

In [199]:
ctr_x

array([ 16,  32,  48,  64,  80,  96, 112, 128, 144, 160, 176, 192, 208, 224])

In [200]:
index = 0
ctr = np.zeros((len(ctr_x) * len(ctr_y), 2))

for x in range(len(ctr_x)):
    for y in range(len(ctr_y)):
        ctr[index, 1] = ctr_x[x] - 8
        ctr[index, 0] = ctr_y[y] - 8
        index +=1

In [201]:
len(ctr) # number of anchors' centers in the featmap 14x14

196

In [202]:
fe_size # grandezza featmap 

14

In [203]:
sub_sample # scaling factor (from 224 -> 14)

16

In [204]:
anchors = np.zeros((fe_size * fe_size * 9, 4))
index = 0
for c in ctr:
    ctr_y, ctr_x = c
    for i in range(len(ratios)):
        for j in range(len(anchor_scales)):
            h = sub_sample * anchor_scales[j] * np.sqrt(ratios[i])
            w = sub_sample * anchor_scales[j] * np.sqrt(1./ ratios[i])
            anchors[index, 0] = ctr_y - h / 2.
            anchors[index, 1] = ctr_x - w / 2.
            anchors[index, 2] = ctr_y + h / 2.
            anchors[index, 3] = ctr_x + w / 2.
            index += 1
print(anchors.shape)
print(anchors)

(1764, 4)
[[    -37.255      -82.51      53.255       98.51]
 [     -82.51     -173.02       98.51      189.02]
 [    -173.02     -354.04      189.02      370.04]
 ...
 [     125.49      170.75      306.51      261.25]
 [     34.981      125.49      397.02      306.51]
 [    -146.04      34.981      578.04      397.02]]


In [205]:
bbox = np.asarray([[20, 30, 100, 200], [50, 100, 150, 200]], dtype=np.float32) # [y1, x1, y2, x2] format
labels = np.asarray([6, 8], dtype=np.int8) # 0 represents background

In [206]:
anchors

array([[    -37.255,      -82.51,      53.255,       98.51],
       [     -82.51,     -173.02,       98.51,      189.02],
       [    -173.02,     -354.04,      189.02,      370.04],
       ...,
       [     125.49,      170.75,      306.51,      261.25],
       [     34.981,      125.49,      397.02,      306.51],
       [    -146.04,      34.981,      578.04,      397.02]])

In [207]:
anchors[:,0]

array([    -37.255,      -82.51,     -173.02, ...,      125.49,      34.981,     -146.04])

In [208]:
inside_index = np.where(
        (anchors[:, 0] >= 0) &
        (anchors[:, 1] >= 0) &
        (anchors[:, 2] <= 224) &
        (anchors[:, 3] <= 224)
    )[0]
print(inside_index.shape) # list of indexes of valid anchors

(68,)


In [209]:
label = np.empty((len(inside_index), ), dtype=np.int32)
label.fill(-1)
print(label.shape)
print(label)

(68,)
[-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1]


In [210]:
valid_anchor_boxes = anchors[inside_index]
print(valid_anchor_boxes.shape)

(68, 4)


In [212]:
ious = np.empty((len(valid_anchor_boxes), 2), dtype=np.float32)
ious.fill(0)
print(bbox)
for num1, i in enumerate(valid_anchor_boxes):
    ya1, xa1, ya2, xa2 = i  
    anchor_area = (ya2 - ya1) * (xa2 - xa1)
    for num2, j in enumerate(bbox):
        yb1, xb1, yb2, xb2 = j
        box_area = (yb2 - yb1) * (xb2 - xb1)
        inter_x1 = max([xb1, xa1])
        inter_y1 = max([yb1, ya1])
        inter_x2 = min([xb2, xa2])
        inter_y2 = min([yb2, ya2])
        if (inter_x1 < inter_x2) and (inter_y1 < inter_y2):
            iter_area = (inter_y2 - inter_y1) * (inter_x2 - inter_x1)
            iou = iter_area / (anchor_area+ box_area - iter_area)
        else:
            iou = 0.
        ious[num1, num2] = iou
print(ious.shape)
print(ious)

[[         20          30         100         200]
 [         50         100         150         200]]
(68, 2)
[[    0.23474   0.0047788]
 [    0.20129   0.0047788]
 [    0.39435     0.13294]
 [    0.36738     0.15801]
 [    0.26922     0.15801]
 [    0.30345    0.069975]
 [     0.1842     0.14713]
 [    0.25816    0.069975]
 [    0.10986      0.1191]
 [   0.044302    0.092415]
 [    0.48259     0.20409]
 [    0.44766     0.24547]
 [    0.32298     0.24547]
 [    0.31837     0.14422]
 [    0.21807     0.22739]
 [    0.27039     0.14422]
 [    0.12858     0.18166]
 [   0.051332     0.13921]
 [     0.7823     0.22489]
 [    0.67201     0.31737]
 [    0.51864     0.28477]
 [    0.45801     0.42496]
 [    0.48025     0.34722]
 [    0.29257     0.47976]
 [    0.34433     0.34722]
 [    0.31837     0.22954]
 [    0.16085     0.36905]
 [    0.23127     0.31973]
 [    0.27039     0.22954]
 [   0.053494     0.26945]
 [    0.13576     0.25161]
 [          0     0.18335]
 [   0.053993     0.19018

### Case 1
find the highest iou for each gt_box and its corresponding anchor box

In [187]:
gt_argmax_ious = ious.argmax(axis=0)
print(gt_argmax_ious)
gt_max_ious = ious[gt_argmax_ious, np.arange(ious.shape[1])]
print(gt_max_ious)


[15  0]
[   0.091736           0]


In [74]:
argmax_ious = ious.argmax(axis=1)
print(argmax_ious.shape)
print(argmax_ious)
max_ious = ious[np.arange(len(inside_index)), argmax_ious]
print(max_ious)

(2235,)
[0 0 0 ... 0 0 0]
[   0.070838    0.070838    0.070838 ...           0           0           0]


In [75]:
gt_argmax_ious = np.where(ious == gt_max_ious)[0]
print(gt_argmax_ious)

[ 604 1390 1398 1513 1521]


In [76]:
pos_iou_threshold  = 0.7
neg_iou_threshold = 0.3

In [77]:
label[max_ious < neg_iou_threshold] = 0

In [78]:
label[gt_argmax_ious] = 1

In [79]:
label[max_ious >= pos_iou_threshold] = 1

Training RPN The Faster_R-CNN paper phrases as follows Each mini-batch arises from a single image that contains many positive and negitive example anchors, but this will bias towards negitive samples as they are dominate. Instead, we randomly sample 256 anchors in an image to compute the loss function of a mini-batch, where the sampled positive and negative anchors have a ratio of up to 1:1. If there are fewer than 128 positive samples in an image, we pad the mini-batch with negitive ones.. From this we can derive two variable as follows

In [80]:
pos_ratio = 0.5
n_sample = 256

In [81]:
n_pos = pos_ratio * n_sample # total positive samples

In [82]:
pos_index = np.where(label == 1)[0]
if len(pos_index) > n_pos:
    disable_index = np.random.choice(pos_index, size=(len(pos_index) - n_pos), replace=False)
    label[disable_index] = -1

In [83]:
n_neg = n_sample * np.sum(label == 1)
neg_index = np.where(label == 0)[0]
if len(neg_index) > n_neg:
    disable_index = np.random.choice(neg_index, size=(len(neg_index) - n_neg), replace = False)
    label[disable_index] = -1

t_{x} = (x - x_{a})/w_{a}
t_{y} = (y - y_{a})/h_{a}
t_{w} = log(w/ w_a)
t_{h} = log(h/ h_a)

In [84]:
max_iou_bbox = bbox[argmax_ious]
print(max_iou_bbox)

[[         20          30         400         500]
 [         20          30         400         500]
 [         20          30         400         500]
 ...
 [         20          30         400         500]
 [         20          30         400         500]
 [         20          30         400         500]]


In [85]:
height = valid_anchor_boxes[:, 2] - valid_anchor_boxes[:, 0]
width = valid_anchor_boxes[:, 3] - valid_anchor_boxes[:, 1]
ctr_y = valid_anchor_boxes[:, 0] + 0.5 * height
ctr_x = valid_anchor_boxes[:, 1] + 0.5 * width
base_height = max_iou_bbox[:, 2] - max_iou_bbox[:, 0]
base_width = max_iou_bbox[:, 3] - max_iou_bbox[:, 1]
base_ctr_y = max_iou_bbox[:, 0] + 0.5 * base_height
base_ctr_x = max_iou_bbox[:, 1] + 0.5 * base_width

In [86]:
eps = np.finfo(height.dtype).eps
height = np.maximum(height, eps)
width = np.maximum(width, eps)
dy = (base_ctr_y - ctr_y) / height
dx = (base_ctr_x - ctr_x) / width
dh = np.log(base_height / height)
dw = np.log(base_width / width)
anchor_locs = np.vstack((dy, dx, dh, dw)).transpose()
print(anchor_locs)


[[    0.49718      2.3091     0.74157      1.6473]
 [    0.32041      2.3091     0.74157      1.6473]
 [    0.14363      2.3091     0.74157      1.6473]
 ...
 [    -3.7969     -3.6172      1.0881      1.3007]
 [    -2.6848     -5.1155     0.74157      1.6473]
 [    -4.0469     -3.6172      1.0881      1.3007]]


In [87]:
anchor_labels = np.empty((len(anchors),), dtype=label.dtype)
anchor_labels.fill(-1)
anchor_labels[inside_index] = label
print(anchor_labels.size)

9216


In [88]:
anchor_locations = np.empty((len(anchors),) + anchors.shape[1:], dtype=anchor_locs.dtype)
anchor_locations.fill(0)
anchor_locations[inside_index, :] = anchor_locs
print(anchor_locations.size)


36864


cnn

In [89]:
mid_channels = 512
in_channels = 1024 # depends on the output feature map. in vgg 16 it is equal to 512
n_anchor = 9 # Number of anchors at each location
conv1 = nn.Conv2d(in_channels, mid_channels, 3, 1, 1)
reg_layer = nn.Conv2d(mid_channels, n_anchor *4, 1, 1, 0)
cls_layer = nn.Conv2d(mid_channels, n_anchor *2, 1, 1, 0) ## I will be going to use softmax here. you can equally use sigmoid if u replace 2 with 1.

In [90]:
# conv sliding layer
conv1.weight.data.normal_(0, 0.01)
conv1.bias.data.zero_()
# Regression layer
reg_layer.weight.data.normal_(0, 0.01)
reg_layer.bias.data.zero_()
# classification layer
cls_layer.weight.data.normal_(0, 0.01)
cls_layer.bias.data.zero_()

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [91]:
x = conv1(out_map) # out_map is obtained in section 1
pred_anchor_locs = reg_layer(x)
pred_cls_scores = cls_layer(x)
print(pred_cls_scores.shape, pred_anchor_locs.shape)
#Out:
#torch.Size([1, 18, 50, 50]) torch.Size([1, 36, 50, 50])

torch.Size([1, 18, 50, 50]) torch.Size([1, 36, 50, 50])


pred_cls_scores and pred_anchor_locs are the output the RPN network and the losses to updates the weights

pred_cls_scores and objectness_scores are used as inputs to the proposal layer, which generate a set of proposal which are further used by RoI network. We will see this in the next section.


In [99]:
pred_anchor_locs.shape

torch.Size([1, 22500, 4])

In [92]:
pred_anchor_locs = pred_anchor_locs.permute(0, 2, 3, 1).contiguous().view(1, -1, 4)
print(pred_anchor_locs.shape)
#Out: torch.Size([1, 22500, 4])
pred_cls_scores = pred_cls_scores.permute(0, 2, 3, 1).contiguous()
print(pred_cls_scores.shape)
#Out torch.Size([1, 50, 50, 18])
objectness_score = pred_cls_scores.view(1, 50, 50, 9, 2)[:, :, :, :, 1].contiguous().view(1, -1)
print(objectness_score.shape)
#Out torch.Size([1, 22500])
pred_cls_scores  = pred_cls_scores.view(1, -1, 2)
print(pred_cls_scores.shape)
# Out torch.size([1, 22500, 2])


torch.Size([1, 22500, 4])
torch.Size([1, 50, 50, 18])
torch.Size([1, 22500])
torch.Size([1, 22500, 2])


The proposal function will take the following parameters

- Weather training_mode or testing mode
- nms_thresh
- n_train_pre_nms — number of bboxes before nms during training
- n_train_post_nms — number of bboxes after nms during training
- n_test_pre_nms — number of bboxes before nms during testing
- n_test_post_nms — number of bboxes after nms during testing
- min_size — minimum height of the object required to create a proposal.


The Faster R_CNN says, RPN proposals highly overlap with each other. To reduced redundancy, we adopt non-maximum supression (NMS) on the proposal regions based on their cls scores. We fix the IoU threshold for NMS at 0.7, which leaves us about 2000 proposal regions per image. After an ablation study, the authors show that NMS does not harm the ultimate detection accuracy, but substantially reduces the number of proposals. After NMS, we use the top-N ranked proposal regions for detection. In the following we training Fast R-CNN using 2000 RPN proposals. During testing they evaluate only 300 proposals, they have tested this with various numbers and obtained this.

In [93]:
nms_thresh = 0.7
n_train_pre_nms = 12000
n_train_post_nms = 2000
n_test_pre_nms = 6000
n_test_post_nms = 300
min_size = 16


convert the loc predictions from the rpn network to bbox [y1, x1, y2, x2] format.

This is the reverse operations of what we have done while assigning ground truth to anchor boxes .This operation decodes predictions by un-parameterizing them and offseting to image. the formulas are as follows

x = (w_{a} * ctr_x_{p}) + ctr_x_{a}
y = (h_{a} * ctr_x_{p}) + ctr_x_{a}
h = np.exp(h_{p}) * h_{a}
w = np.exp(w_{p}) * w_{a}
and later convert to y1, x1, y2, x2 format


In [98]:
anchors.shape

(9216, 4)

In [94]:
anc_height = anchors[:, 2] - anchors[:, 0]
anc_width = anchors[:, 3] - anchors[:, 1]
anc_ctr_y = anchors[:, 0] + 0.5 * anc_height
anc_ctr_x = anchors[:, 1] + 0.5 * anc_width

In [100]:
anc_height.shape

(9216,)

In [96]:
anc_ctr_y[:, np.newaxis].shape

(9216, 1)

In [101]:
pred_anchor_locs_numpy = pred_anchor_locs[0].data.numpy()
objectness_score_numpy = objectness_score[0].data.numpy()
dy = pred_anchor_locs_numpy[:, 0::4]
print(dy.shape)
dx = pred_anchor_locs_numpy[:, 1::4]
print(dx)
dh = pred_anchor_locs_numpy[:, 2::4]
dw = pred_anchor_locs_numpy[:, 3::4]
ctr_y = dy * anc_height[:, np.newaxis] + anc_ctr_y[:, np.newaxis]
ctr_x = dx * anc_width[:, np.newaxis] + anc_ctr_x[:, np.newaxis]
h = np.exp(dh) * anc_height[:, np.newaxis]
w = np.exp(dw) * anc_width[:, np.newaxis]

(22500, 1)
[[  -0.033429]
 [   0.011463]
 [   0.011519]
 ...
 [  0.0092935]
 [    -0.0558]
 [  -0.016605]]


ValueError: operands could not be broadcast together with shapes (22500,1) (9216,1) 

In [ ]:
roi = np.zeros(pred_anchor_locs_numpy.shape, dtype=pred_anchor_locs_numpy.dtype)
roi[:, 0::4] = ctr_y - 0.5 * h
roi[:, 1::4] = ctr_x - 0.5 * w
roi[:, 2::4] = ctr_y + 0.5 * h
roi[:, 3::4] = ctr_x + 0.5 * w

In [ ]:
img_size = (800, 800) #Image size
roi[:, slice(0, 4, 2)] = np.clip(
            roi[:, slice(0, 4, 2)], 0, img_size[0])
roi[:, slice(1, 4, 2)] = np.clip(
    roi[:, slice(1, 4, 2)], 0, img_size[1])
print(roi)

[[          0           0       55.47      96.655]
 [          0           0      105.52       164.5]
 [          0           0      199.96      318.56]
 ...
 [     696.78      747.85         800         800]
 [     629.12      711.55         800         800]
 [     417.17      611.86         800         800]]


In [ ]:
hs = roi[:, 2] - roi[:, 0]
ws = roi[:, 3] - roi[:, 1]
keep = np.where((hs >= min_size) & (ws >= min_size))[0]
roi = roi[keep, :]
score = objectness_score_numpy[keep]
print(score.shape)

(22500,)


In [ ]:
order = score.ravel().argsort()[::-1]
print(order)

[    0   888   879 ... 22037 21604 21614]


In [ ]:
order = order[:n_train_pre_nms]
roi = roi[order, :]
print(roi.shape)
print(roi)

(12000, 4)
[[          0           0       55.47      96.655]
 [     690.16           0         800      77.854]
 [     682.62           0         800      75.834]
 ...
 [     36.678       315.6      401.29         800]
 [     308.68       315.6      673.29         800]
 [     645.97      710.75      778.33         800]]


Apply non-maximum supression

In [ ]:
y1 = roi[:, 0]
x1 = roi[:, 1]
y2 = roi[:, 2]
x2 = roi[:, 3]
area = (x2 - x1 + 1) * (y2 - y1 + 1)
order = score.argsort()[::-1]
keep = []
while order.size > 0:
    i = order[0]
    xx1 = np.maximum(x1[i], x1[order[1:]])
    yy1 = np.maximum(y1[i], y1[order[1:]])
    xx2 = np.minimum(x2[i], x2[order[1:]])
    yy2 = np.minimum(y2[i], y2[order[1:]])
w = np.maximum(0.0, xx2 - xx1 + 1)
h = np.maximum(0.0, yy2 - yy1 + 1)
inter = w * h
ovr = inter / (area[i] + area[order[1:]] - inter)
inds = np.where(ovr <= nms_thresh)[0]
order = order[inds + 1]
keep = keep[:n_train_post_nms] # while training/testing , use accordingly
roi = roi[keep] # the final region proposals